# Exploration des données brutes Olist

**Objectif :** comprendre les données **avant** toute transformation, pour justifier les choix du pipeline dbt.

Chaque section part d'une **observation** dans les CSV bruts et se termine par le **choix** qu'elle justifie.
Les choix sont résumés dans `docs/choix_de_modelisation.md`.

> On lit directement les fichiers de `ctc_olist/data/raw/`, sans passer par dbt.
> Pour les obtenir : `./scripts/download_data.sh`

## Préparation

DuckDB peut lire un CSV comme une table. La petite fonction `sql()` lance une requête et affiche le résultat en tableau.

In [ ]:
import duckdb

# Dossier des CSV bruts (le notebook est dans notebooks/)
RAW = "../ctc_olist/data/raw"

def sql(query):
    """Lance une requête SQL et renvoie un tableau pandas."""
    return duckdb.sql(query).df()

def csv(name):
    """Chemin d'un fichier brut, à utiliser dans une requête."""
    return f"'{RAW}/{name}.csv'"

## 1. Vue d'ensemble des fichiers

**Question :** combien de fichiers, de lignes et de colonnes ? Quels types DuckDB devine-t-il ?

In [ ]:
files = [
    "olist_orders_dataset",
    "olist_customers_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "product_category_name_translation",
    "olist_geolocation_dataset",
]

# Nombre de lignes et de colonnes de chaque fichier
import pandas as pd

rows = []
for f in files:
    n_rows = sql(f"select count(*) as n from {csv(f)}")["n"][0]
    n_cols = len(sql(f"select * from {csv(f)} limit 0").columns)
    rows.append({"file": f, "rows": n_rows, "columns": n_cols})

pd.DataFrame(rows)

**À retenir :** 9 fichiers. La géolocalisation est le plus gros, mais elle ne sert pas à notre sujet (la satisfaction client) : on ne la déclare pas comme source.

### Les types devinés par DuckDB

Exemple avec les commandes et les clients.

In [ ]:
sql(f"describe select * from {csv('olist_orders_dataset')}")

In [ ]:
sql(f"describe select * from {csv('olist_customers_dataset')}")

In [ ]:
# Des codes postaux commencent par 0 : en nombre, ce zéro disparaîtrait
sql(f"""
    select customer_zip_code_prefix
    from {csv('olist_customers_dataset')}
    where customer_zip_code_prefix like '0%'
    limit 5
""")

**À retenir :**
- Les types sont **devinés** à partir du contenu du fichier : ils pourraient changer si le fichier changeait.
- Certains noms ne correspondent pas au contenu : `order_delivered_carrier_date` contient une **heure**,
  et `order_estimated_delivery_date` est lue comme date et heure alors qu'elle est toujours à minuit.
- Les codes postaux commencent parfois par **0** : il faut les garder en **texte**.

**→ Choix :** en staging, on **convertit explicitement** chaque type (`cast`), on renomme avec la convention `_at` (date et heure) / `_date` (jour), et on garde les codes postaux en texte.

## 2. Clés et grains

**Question :** dans chaque fichier, quelle colonne identifie une ligne de façon unique ?

Méthode : si le nombre de valeurs **distinctes** d'une colonne est égal au nombre de **lignes**, la colonne est unique.

In [ ]:
# Pour chaque candidat de clé : nombre de lignes et nombre de valeurs distinctes
checks = [
    ("olist_orders_dataset",              "order_id"),
    ("olist_customers_dataset",           "customer_id"),
    ("olist_customers_dataset",           "customer_unique_id"),
    ("olist_order_items_dataset",         "order_id"),
    ("olist_order_items_dataset",         "(order_id, order_item_id)"),
    ("olist_order_payments_dataset",      "(order_id, payment_sequential)"),
    ("olist_order_reviews_dataset",       "review_id"),
    ("olist_order_reviews_dataset",       "order_id"),
    ("olist_order_reviews_dataset",       "(review_id, order_id)"),
    ("olist_products_dataset",            "product_id"),
    ("olist_sellers_dataset",             "seller_id"),
    ("product_category_name_translation", "product_category_name"),
]

results = []
for file, key in checks:
    r = sql(f"select count(*) as n_rows, count(distinct {key}) as n_distinct from {csv(file)}")
    n_rows, n_distinct = r["n_rows"][0], r["n_distinct"][0]
    results.append({"file": file, "key": key, "rows": n_rows,
                    "distinct": n_distinct, "unique": n_rows == n_distinct})

pd.DataFrame(results)

**À retenir :**
- Commandes, produits, vendeurs : **une colonne suffit** (`order_id`, `product_id`, `seller_id`).
- Articles et paiements : il faut **deux colonnes** (clé composée).
- Avis : ni `review_id` ni `order_id` ne sont uniques seuls. Seule la paire l'est.
- Clients : `customer_id` est unique, mais **pas** `customer_unique_id`.

### Un client, plusieurs `customer_id`

**Question :** combien de `customer_id` différents pour une même personne ?

In [ ]:
# Nombre de customer_id par personne, puis nombre de personnes dans chaque cas
sql(f"""
    with per_person as (
        select customer_unique_id, count(*) as nb_customer_id
        from {csv('olist_customers_dataset')}
        group by customer_unique_id
    )
    select nb_customer_id, count(*) as nb_persons
    from per_person
    group by nb_customer_id
    order by nb_customer_id
""")

In [ ]:
# Chaque commande a son propre customer_id
sql(f"""
    select
        count(*)                    as orders,
        count(distinct customer_id) as distinct_customer_id
    from {csv('olist_orders_dataset')}
""")

**À retenir :** Olist crée un **nouveau `customer_id` à chaque commande** (99 441 commandes = 99 441 `customer_id`).
Une même personne qui commande plusieurs fois a donc plusieurs `customer_id`, mais un seul `customer_unique_id`.

**→ Choix :**
- Chaque modèle de staging garde le **grain de sa source**, avec une clé composée quand il le faut (articles, paiements, avis).
- `dim_customers` a pour grain **la personne** (`customer_unique_id`), sinon un client fidèle compterait pour plusieurs clients.

## 3. Les clients avec plusieurs adresses

**Question :** une même personne a-t-elle toujours la même adresse d'une commande à l'autre ?

Chaque `customer_id` a sa propre adresse. Comme une personne peut avoir plusieurs `customer_id` (section 2), elle peut aussi avoir plusieurs adresses.

In [ ]:
# Nombre d'adresses différentes (code postal + ville + état) par personne
sql(f"""
    with addresses as (
        select
            customer_unique_id,
            count(distinct customer_zip_code_prefix || customer_city || customer_state) as nb_addresses
        from {csv('olist_customers_dataset')}
        group by customer_unique_id
    )
    select
        count(*)                      as persons,
        cast(count_if(nb_addresses > 1) as integer) as persons_with_several_addresses
    from addresses
""")

### Un exemple : les adresses d'une personne dans le temps

In [ ]:
# Une personne avec plusieurs adresses, commande par commande
sql(f"""
    with movers as (
        select customer_unique_id
        from {csv('olist_customers_dataset')}
        group by customer_unique_id
        having count(distinct customer_city) > 1
        order by customer_unique_id
        limit 1
    )
    select
        c.customer_unique_id,
        o.order_purchase_timestamp,
        c.customer_city,
        c.customer_state
    from {csv('olist_customers_dataset')} c
    join {csv('olist_orders_dataset')} o on c.customer_id = o.customer_id
    where c.customer_unique_id in (select customer_unique_id from movers)
    order by o.order_purchase_timestamp
""")

**À retenir :**
- Seulement **252 personnes sur 96 096** (0,3 %) ont plusieurs adresses.
- Dans l'exemple, la personne commande en mai 2017 puis en juillet 2018, dans deux villes différentes.
- Ce n'est pas toujours un déménagement : l'adresse est celle de la **livraison** (un cadeau envoyé ailleurs, par exemple).
- Il faut quand même choisir **quelle adresse** garder dans la dimension client.

**→ Choix :** `dim_customers` est une **SCD de type 1** : on garde seulement la **dernière adresse connue** (celle de la commande la plus récente) et on écrase les anciennes.

Le type 2 (garder l'historique, une ligne par adresse) serait plus complet, mais inutile ici : les déménagements sont trop rares pour changer l'analyse.